第七课

Use AI (机器学习 / 深度学习: 随机森林、XGBoost、长短期记忆网络) to predict future cash flows

Use DCF -> 估值

In [1]:
import rqdatac as rq
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import RFECV
from sklearn.metrics import r2_score, mean_absolute_error

# Log in to rqdatac
rq.init()

# Get Shanghai and Shenzhen 300 constituent stocks
constituents = rq.index_components('000300.XSHG')

print(f"number of constituents: {len(constituents)}")

start_date = '2010-01-01'
end_date = '2025-12-31'

factors = [
    # revenue
    'revenue',
    'operating_revenue',
    'net_operating_revenue',
    
    # valuation
    'pe_ratio',
    'pb_ratio',
    'ps_ratio',
    'pcf_ratio',
    'book_value_per_share',
    'book_value_per_share_lf',
    'book_value_per_share_ttm',

    # profitability
    'return_on_equity',
    'return_on_asset',
    "profit_before_tax",
    "minority_profit",
    "net_profit",
    "net_profit_parent_company",
    "ebit",

    # leverage/liquidity
    'debt_to_equity',
    'current_ratio',
    
    # growth
    'revenue_growth_rate',
    'net_profit_growth_rate',
    'peg_ratio_ttm',

    # free cash flow
    'free_cash_flow_company_per_share',
    'free_cash_flow_company_per_share_ttm',

    # Balance
    "cash_equivalent", 
    "financial_asset_held_for_trading", 
    "current_assets", 
    "bill_receivable", 
    "net_accts_receivable", 
    "other_accts_receivable", 
    "inventory", 
    "deferred_expense", 
    "other_current_assets", 
    "long_term_equity_investment", 
    "net_long_term_equity_investment", 
    "real_estate_investment", 
    "net_fixed_assets", 
    "construction_in_progress", 
    "fixed_asset_to_be_disposed", 
    "intangible_assets", 
    "long_term_deferred_expenses", 
    "deferred_income_tax_assets", 
    "other_non_current_assets", 
    "short_term_loans", 
    "notes_payable", 
    "accts_payable", 
    "long_term_liabilities_due_one_year", 
    "non_current_liability_due_one_year", 
    "other_current_liabilities", 
    "current_liabilities", 
    "long_term_loans", 
    "bond_payable", 
    "long_term_payable", 
    "deferred_income_tax_liabilities", 
    "deferred_revenue", 
    "other_non_current_liabilities", 
    "total_assets", 
    "minority_interest", 
    "equity_parent_company", 
    "total_equity",
]

# Fetch financial statement ratio information
financial_data = rq.get_factor(constituents, factors, start_date=start_date, end_date=end_date)

financial_data = financial_data.fillna(0, inplace=False)

print("financial data:")
print(financial_data.head(5))

number of constituents: 300
financial data:
                               revenue  operating_revenue  \
order_book_id date                                          
600161.XSHG   2010-01-04  4.694512e+08       4.694512e+08   
              2010-01-05  4.694512e+08       4.694512e+08   
              2010-01-06  4.694512e+08       4.694512e+08   
              2010-01-07  4.694512e+08       4.694512e+08   
              2010-01-08  4.694512e+08       4.694512e+08   

                          net_operating_revenue  pe_ratio  pb_ratio  ps_ratio  \
order_book_id date                                                              
600161.XSHG   2010-01-04                    0.0  139.6810   18.0140   20.4454   
              2010-01-05                    0.0  138.4278   17.8524   20.2619   
              2010-01-06                    0.0  136.7046   17.6302   20.0097   
              2010-01-07                    0.0  131.0652   16.9029   19.1843   
              2010-01-08                  

In [2]:
# financial_data_reset = financial_data.reset_index()

# financial_data_reset['date'] = pd.to_datetime(
#     financial_data_reset['date']
# )

# financial_data_yearly = (
#     financial_data_reset
#     .groupby([
#         'order_book_id',
#         pd.Grouper(key='date', freq='YE')
#     ])
#     .last()
#     .reset_index()
# )

# financial_data = financial_data_yearly

# print(financial_data.head(5))

X = financial_data

X = X.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='YE')
]).last().fillna(0)

X = X.reorder_levels(['order_book_id', 'date']).shift(-1)

print(X.head(5))

                               revenue  operating_revenue  \
order_book_id date                                          
000001.XSHE   2010-12-31  2.070140e+10       2.070140e+10   
              2011-12-31  2.953167e+10       2.953167e+10   
              2012-12-31  3.734500e+10       3.734500e+10   
              2013-12-31  5.465100e+10       5.465100e+10   
              2014-12-31  7.115200e+10       7.115200e+10   

                          net_operating_revenue  pe_ratio  pb_ratio  ps_ratio  \
order_book_id date                                                              
000001.XSHE   2010-12-31                    0.0    8.6470    1.1458    3.1244   
              2011-12-31                    0.0    6.3975    1.0043    2.1333   
              2012-12-31                    0.0    6.7572    1.0580    2.1113   
              2013-12-31                    0.0    9.4113    1.4279    2.6041   
              2014-12-31                    0.0    7.8525    1.0918    1.9082   

    

In [3]:
# Fetch next year's free cash flow
next_year_free_cash_flow = rq.get_factor(constituents, 'free_cash_flow_company_per_share_ttm', start_date='2011-01-01', end_date=end_date)

next_year_free_cash_flow = next_year_free_cash_flow.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='YE')
]).last().fillna(0)

next_year_free_cash_flow = next_year_free_cash_flow.reorder_levels(['order_book_id', 'date']).shift(-1)

next_year_free_cash_flow = next_year_free_cash_flow.fillna(0, inplace=False)

y = next_year_free_cash_flow

print(next_year_free_cash_flow)

                          free_cash_flow_company_per_share_ttm
order_book_id date                                            
000001.XSHE   2011-12-31                              4.734325
              2012-12-31                              4.077166
              2013-12-31                              2.126234
              2014-12-31                              1.859431
              2015-12-31                              1.384475
...                                                        ...
688981.XSHG   2021-12-31                              1.428541
              2022-12-31                              1.360557
              2023-12-31                              0.078630
              2024-12-31                             -0.462069
              2025-12-31                              0.000000

[3712 rows x 1 columns]


In [4]:
# Remove 2025 data because 2026 has not finished yet
X = X[X.index.get_level_values('date') != end_date]
print(X)

                               revenue  operating_revenue  \
order_book_id date                                          
000001.XSHE   2010-12-31  2.070140e+10       2.070140e+10   
              2011-12-31  2.953167e+10       2.953167e+10   
              2012-12-31  3.734500e+10       3.734500e+10   
              2013-12-31  5.465100e+10       5.465100e+10   
              2014-12-31  7.115200e+10       7.115200e+10   
...                                ...                ...   
688981.XSHG   2020-12-31  2.537100e+10       2.537100e+10   
              2021-12-31  3.776356e+10       3.776356e+10   
              2022-12-31  3.309824e+10       3.309824e+10   
              2023-12-31  4.187872e+10       4.187872e+10   
              2024-12-31  4.951042e+10       4.951042e+10   

                          net_operating_revenue  pe_ratio  pb_ratio  ps_ratio  \
order_book_id date                                                              
000001.XSHE   2010-12-31                    

In [5]:
# Align X and y
# X, y = X.align(y, join='inner', axis=0)

# print(f"X shape: {X.shape}")
# print(f"y shape: {y.shape}")
# print('X:')
# print(X)
# print()
# print('*' * 60)
# print('y:')
# print(y)

import numpy as np

# Assuming your data is already multi-indexed with (order_book_id, date)
# Align features (year t) with target (year t+1)

# Method 1: Shift per stock
def align_features_target(X, y):
    """
    X: features at time t
    y: target at time t
    Returns: X_aligned (time t), y_aligned (time t+1)
    """
    # Reset index to work with dates
    X_df = X.reset_index()
    y_df = y.reset_index()
    
    # Rename target column for clarity
    y_df = y_df.rename(columns={'free_cash_flow_company_per_share_ttm': 'target'})
    
    # Create a shifted version of y for each stock
    aligned_pairs = []
    
    for stock in X_df['order_book_id'].unique():
        X_stock = X_df[X_df['order_book_id'] == stock].copy()
        y_stock = y_df[y_df['order_book_id'] == stock].copy()
        
        # Sort by date
        X_stock = X_stock.sort_values('date')
        y_stock = y_stock.sort_values('date')
        
        # Shift y backwards by 1 year (so y at t-1 pairs with X at t)
        # Or more intuitively: align X at year t with y at year t+1
        y_stock['target_next_year'] = y_stock['target'].shift(-1)
        
        # Merge on date
        merged = X_stock.merge(
            y_stock[['order_book_id', 'date', 'target_next_year']], 
            on=['order_book_id', 'date']
        )
        
        # Drop rows where next year's target is missing
        merged = merged.dropna(subset=['target_next_year'])
        
        aligned_pairs.append(merged)
    
    # Combine all stocks
    aligned_df = pd.concat(aligned_pairs, ignore_index=True)
    
    # Separate X and y
    X_aligned = aligned_df.drop(['order_book_id', 'date', 'target_next_year'], axis=1)
    y_aligned = aligned_df['target_next_year']
    
    return X_aligned, y_aligned, aligned_df[['order_book_id', 'date']]  # Keep metadata

def robust_align_features_target(X, y):
    """
    Robust alignment handling index mismatches
    """
    # Reset indices to columns
    X_df = X.reset_index()
    y_df = y.reset_index()
    
    # Rename target column
    y_df = y_df.rename(columns={'free_cash_flow_company_per_share_ttm': 'target'})
    
    # Sort both by stock and date
    X_df = X_df.sort_values(['order_book_id', 'date'])
    y_df = y_df.sort_values(['order_book_id', 'date'])
    
    aligned_pairs = []
    
    for stock in X_df['order_book_id'].unique():
        X_stock = X_df[X_df['order_book_id'] == stock].copy()
        y_stock = y_df[y_df['order_book_id'] == stock].copy()
        
        if len(y_stock) == 0:
            print(f"Warning: No target data for {stock}")
            continue
        
        # Ensure both have consistent date ranges
        common_dates = set(X_stock['date']).intersection(set(y_stock['date']))
        if len(common_dates) == 0:
            print(f"Warning: No common dates for {stock}")
            continue
        
        X_stock = X_stock[X_stock['date'].isin(common_dates)]
        y_stock = y_stock[y_stock['date'].isin(common_dates)]
        
        # Sort by date
        X_stock = X_stock.sort_values('date')
        y_stock = y_stock.sort_values('date')
        
        # Shift target to next year
        y_stock['target_next_year'] = y_stock['target'].shift(-1)
        
        # Merge on date
        merged = X_stock.merge(
            y_stock[['order_book_id', 'date', 'target_next_year']],
            on=['order_book_id', 'date'],
            how='inner'
        )
        
        # Drop rows with NaN target (last year of each stock)
        merged = merged.dropna(subset=['target_next_year'])
        
        if len(merged) > 0:
            aligned_pairs.append(merged)
    
    if len(aligned_pairs) == 0:
        raise ValueError("No aligned data found. Check your date ranges.")
    
    # Combine all stocks
    aligned_df = pd.concat(aligned_pairs, ignore_index=True)
    
    # Separate X, y, and metadata
    feature_cols = [col for col in X_df.columns if col not in ['order_book_id', 'date']]
    X_aligned = aligned_df[feature_cols]
    y_aligned = aligned_df['target_next_year']
    metadata = aligned_df[['order_book_id', 'date']]
    
    return X_aligned, y_aligned, metadata

# Apply the robust alignment
try:
    X_aligned, y_aligned, metadata = robust_align_features_target(X, y)
    print(f"Success! Aligned {len(X_aligned)} samples")
    print(f"X shape: {X_aligned.shape}")
    print(f"y shape: {y_aligned.shape}")
    print(f"Date range: {metadata['date'].min()} to {metadata['date'].max()}")
except ValueError as e:
    print(f"Alignment failed: {e}")

# # Apply alignment
# X_aligned, y_aligned, metadata = align_features_target(X, y)

# print(f"Original X shape: {X.shape}")
# print(f"Original y shape: {y.shape}")
# print(f"Aligned X shape: {X_aligned.shape}")
# print(f"Aligned y shape: {y_aligned.shape}")

Success! Aligned 3113 samples
X shape: (3113, 60)
y shape: (3113,)
Date range: 2011-12-31 00:00:00 to 2023-12-31 00:00:00


In [6]:
print(X_aligned.head(20))

         revenue  operating_revenue  net_operating_revenue  pe_ratio  \
0   2.953167e+10       2.953167e+10                    0.0    6.3975   
1   3.734500e+10       3.734500e+10                    0.0    6.7572   
2   5.465100e+10       5.465100e+10                    0.0    9.4113   
3   7.115200e+10       7.115200e+10                    0.0    7.8525   
4   8.196800e+10       8.196800e+10                    0.0    6.8399   
5   7.983300e+10       7.983300e+10                    0.0    9.9148   
6   8.666400e+10       8.666400e+10                    0.0    6.5760   
7   1.029580e+11       1.029580e+11                    0.0   11.4079   
8   1.165640e+11       1.165640e+11                    0.0   13.9148   
9   1.271900e+11       1.271900e+11                    0.0    8.9670   
10  1.382650e+11       1.382650e+11                    0.0    6.2272   
11  1.276340e+11       1.276340e+11                    0.0    3.9923   
12  1.115820e+11       1.115820e+11                    0.0    4.

In [7]:
print(y_aligned.head(20))

0     4.077166
1     2.126234
2     1.859431
3     1.384475
4     0.981718
5     1.520333
6     0.805708
7     2.159187
8     2.228611
9     1.980620
10    3.052007
11    1.533604
12    2.002559
13    0.542117
14    1.594830
15    1.217993
16    2.933680
17    1.477566
18    3.296967
19    3.636659
Name: target_next_year, dtype: float64


## 随机森林

In [8]:
## 随机森林模型

# Train test split 80/20
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned.iloc[:split], X_aligned.iloc[split:]
y_train, y_test = y_aligned.iloc[:split], y_aligned.iloc[split:]

# Define hyperparameters once
rf_params = dict(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=5,
    max_features='sqrt'
)

# Feature selection
model = RandomForestRegressor(**rf_params)
selector = RFECV(estimator=model, cv=5)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)  # apply same selection to test set

# Cross-validate, then final fit
rf_model = RandomForestRegressor(**rf_params)
cv_scores = cross_val_score(rf_model, X_train_selected, y_train, cv=5, scoring='r2')
print(f"Cross-validation R^2 scores: {cv_scores}")

# Fit the model
rf_model.fit(X_train_selected, y_train)

# Evaluate
y_pred = rf_model.predict(X_test_selected)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"Test R^2: {r2}, Test MAE: {mae}")

Cross-validation R^2 scores: [0.56872933 0.37922285 0.56461271 0.30349342 0.56980986]
Test R^2: -0.3645416881444312, Test MAE: 0.8199640966050742


## XGBoost

In [11]:
# XGBoost Model

from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score

# Train-test split
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned.iloc[:split], X_aligned.iloc[split:]
y_train, y_test = y_aligned.iloc[:split], y_aligned.iloc[split:]

# XGBoost model
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

# Cross validation
xgb_cv_scores = cross_val_score(
    xgb_model,
    X_train,
    y_train.values.ravel(),
    cv=5,
    scoring='r2'
)

print("XGBoost Cross-validation R^2 scores:")
print(xgb_cv_scores)
print(f"Average CV R^2: {xgb_cv_scores.mean()}")

# Train model
xgb_model.fit(X_train, y_train.values.ravel())

# Prediction
xgb_pred = xgb_model.predict(X_test)

# Evaluation
xgb_r2 = r2_score(y_test, xgb_pred)
xgb_mae = mean_absolute_error(y_test, xgb_pred)

print(f"XGBoost Test R^2: {xgb_r2}")
print(f"XGBoost Test MAE: {xgb_mae}")

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/tylerpruitt/Desktop/量化投资交易/Quant Trading Desk/.venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <5A4ADBF3-2060-3ECB-AB2B-F8A248DDA0A7> /Users/tylerpruitt/Desktop/量化投资交易/Quant Trading Desk/.venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryp

## 长短期记忆

In [ ]:
# 长短期记忆模型

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Train-test split
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned.iloc[:split], X_aligned.iloc[split:]
y_train, y_test = y_aligned.iloc[:split], y_aligned.iloc[split:]

# Scale features
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

# Reshape for LSTM
# Shape: (samples, timesteps, features)
X_train_lstm = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    1,
    X_train_scaled.shape[1]
)

X_test_lstm = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    1,
    X_test_scaled.shape[1]
)

# Build LSTM model
lstm_model = Sequential()

lstm_model.add(
    LSTM(
        units=64,
        return_sequences=False,
        input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])
    )
)

lstm_model.add(Dropout(0.2))

lstm_model.add(Dense(32, activation='relu'))

lstm_model.add(Dense(1))

# Compile
lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse'
)

# Train
history = lstm_model.fit(
    X_train_lstm,
    y_train_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Predict
lstm_pred_scaled = lstm_model.predict(X_test_lstm)

# Inverse transform
lstm_pred = y_scaler.inverse_transform(lstm_pred_scaled)

# Evaluation
lstm_r2 = r2_score(y_test, lstm_pred)
lstm_mae = mean_absolute_error(y_test, lstm_pred)

print(f"LSTM Test R^2: {lstm_r2}")
print(f"LSTM Test MAE: {lstm_mae}")

## 预测

In [ ]:
# =========================
# Company Prediction
# =========================

# Random Forest Prediction
rf_prediction = rf_model.predict(
    selector.transform(ly_financial_df)
)

# XGBoost Prediction
xgb_prediction = xgb_model.predict(
    ly_financial_df
)

# LSTM Prediction

# Scale
ly_scaled = x_scaler.transform(ly_financial_df)

# Reshape
ly_lstm = ly_scaled.reshape(
    ly_scaled.shape[0],
    1,
    ly_scaled.shape[1]
)

# Predict
lstm_prediction_scaled = lstm_model.predict(ly_lstm)

# Inverse scale
lstm_prediction = y_scaler.inverse_transform(
    lstm_prediction_scaled
)

print(f"Random Forest Prediction: {rf_prediction}")
print(f"XGBoost Prediction: {xgb_prediction}")
print(f"LSTM Prediction: {lstm_prediction}")

## 模型结果

In [ ]:
# =========================
# Model Comparison
# =========================

comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'LSTM'],
    'R2': [
        r2,
        xgb_r2,
        lstm_r2
    ],
    'MAE': [
        mae,
        xgb_mae,
        lstm_mae
    ]
})

print(comparison_df)

In [ ]:
import datetime

def get_today_date() -> str:
    """Get today's date in the format 'YYYY-MM-DD'."""
    today = datetime.date.today()
    return today.strftime("%Y-%m-%d")

def get_order_book_id(company_name):
    """Function to map company name to order_book_id."""
    all_stocks = rq.all_instruments(type='CS')
    stock = all_stocks[all_stocks['symbol'] == company_name]
    
    print('*' * 60)
    print(f'function get_order_book_id(company_name={company_name})')
    print()
    print(f"stock: {stock['order_book_id'].values if not stock.empty else None}")

    if not stock.empty:
        print(f'\nreturning {stock['order_book_id'].values[0]}', end=f"\n{'*' * 60}\n\n")
        return stock['order_book_id'].values[0]
    else:
        stock = all_stocks[all_stocks['abbrev_symbol'].str.contains(company_name)]
        if not stock.empty:
            print(f'\nreturning {stock['order_book_id'].values[0]}', end=f"\n{'*' * 60}\n\n")
            return stock['order_book_id'].values[0]
        else:
            print('\nreturning None', end=f"\n{'*' * 60}\n\n")
            return None

In [ ]:
# Input company name
company_name = input("Enter the name of the listed company: ")
order_book_id = get_order_book_id(company_name)

prediction_start_date = '2025-01-01'
prediction_end_date = '2025-12-31'

if not order_book_id:
    while not order_book_id:
        print("Company not found. Please try again.")

        # Input company name
        company_name = input("Enter the name of the listed company: ")
        order_book_id = get_order_book_id(company_name)

# Get financial data for the company
print(f"Fetching financial data for {company_name} from {prediction_start_date} to {prediction_end_date}...", end='')
ly_financial_data = rq.get_factor([order_book_id], factors, start_date=prediction_start_date, end_date=prediction_end_date)
print("Finished.")

ly_financial_df = ly_financial_data.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='YE')
]).last().fillna(0)

print(f'\n{company_name} financial data:')
print(ly_financial_df)

In [ ]:
# Make prediction from financial data from company
prediction_df = rf_model.predict(selector.transform(ly_financial_df))
print(f"Prediction for {company_name}: {prediction_df}", end='\n\n')

years = ly_financial_df.index.get_level_values('date').year

# Create df that associates next year with predictions
ny_prediction_df = pd.DataFrame(
    {
        'Year': years+1,
        'Price Upside/Downside Prediction (%)': prediction_df[0][0],
        'Predicted P/E': prediction_df[0][1],
        'Predicted P/B': prediction_df[0][2],
        'Predicted P/S': prediction_df[0][3],
        'Predicted PCF': prediction_df[0][4],
        'Predicted Return on Equity': prediction_df[0][5],
        'Predicted Return on Asset': prediction_df[0][6],
        'Predicted Debt/Equity': prediction_df[0][7],
        'Predicted Current Ratio': prediction_df[0][8],
        'Predicted Revenue Growth Rate': prediction_df[0][9],
        'Predicted Net Profit Growth Rate': prediction_df[0][10],
        'Predicted PEG Ratio TTM': prediction_df[0][11],
        'revenue': prediction_df[0][12],
        'profit before tax': prediction_df[0][13],
        'minority profit': prediction_df[0][14],
        'net profit': prediction_df[0][15],
        'net profit parent company': prediction_df[0][16],
        'ebit': prediction_df[0][17],
        'cash equivalent': prediction_df[0][18],
        'financial asset held for trading': prediction_df[0][19],
        'current assets': prediction_df[0][20],
        'bill receivable': prediction_df[0][21],
        'net accts receivable': prediction_df[0][22],
        'other accts receivable': prediction_df[0][23],
        'inventory': prediction_df[0][24],
        'deferred expense': prediction_df[0][25],
        'other current assets': prediction_df[0][26],
        'long term equity investment': prediction_df[0][27],
        'net long term equity investment': prediction_df[0][28],
        'real estate investment': prediction_df[0][29],
        'net fixed assets': prediction_df[0][30],
        'construction in progress': prediction_df[0][31],
        'fixed asset to be disposed': prediction_df[0][32],
        'intangible assets': prediction_df[0][33],
        'long term deferred expenses': prediction_df[0][34],
        'deferred income tax assets': prediction_df[0][35],
        'other non current assets': prediction_df[0][36],
        'short term loans': prediction_df[0][37],
        'notes payable': prediction_df[0][38],
        'accts payable': prediction_df[0][39],
        'long term liabilities due one year': prediction_df[0][40],
        'non current liability due one year': prediction_df[0][41],
        'other current liabilities': prediction_df[0][42],
        'current liabilities': prediction_df[0][43],
        'long term loans': prediction_df[0][44],
        'bond payable': prediction_df[0][45],
        'long term payable': prediction_df[0][46],
        'deferred income tax liabilities': prediction_df[0][47],
        'deferred revenue': prediction_df[0][48],
        'other non current liabilities': prediction_df[0][49],
        'total assets': prediction_df[0][50],
        'minority interest': prediction_df[0][51],
        'equity parent company': prediction_df[0][52],
        'total equity': prediction_df[0][53],
    }
)
print(f"Prediction results for {company_name}:\n{ny_prediction_df}", end='\n\n')


In [ ]:
# Fetch YTD returns for the stock
company_YTD_prices = rq.get_price(order_book_id, start_date=prediction_end_date, end_date=get_today_date(), fields='close', expect_df=False, adjust_type='none')

current_YTD_gain = company_YTD_prices.iloc[-1] - company_YTD_prices.iloc[0]
current_YTD_return = current_YTD_gain / company_YTD_prices.iloc[0]

ny_prediction_df['Predicted EOY Price (RMB)'] = company_YTD_prices.iloc[-1] * (1 + ny_prediction_df['Price Upside/Downside Prediction (%)'])
ny_prediction_df['Predicted Change By EOY (RMB)'] = ny_prediction_df['Predicted EOY Price (RMB)'] - company_YTD_prices.iloc[-1]
ny_prediction_df['Predicted Change By EOY (%)'] = ny_prediction_df['Price Upside/Downside Prediction (%)'] - current_YTD_return
ny_prediction_df['Current Price (RMB)'] = company_YTD_prices.iloc[-1]
ny_prediction_df['Current YTD Return (%)'] = current_YTD_return
ny_prediction_df['Current YTD Return (RMB)'] = current_YTD_gain
print(ny_prediction_df)

In [ ]:
# Generate report
report = f"Financial data for {company_name}:\n\n{ly_financial_df}\n\n{'*' * 60}\n\nPrediction result:\n\n{ny_prediction_df}"
print(report)